# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset using the `mlcroissant` library. We'll walk through loading metadata, inspecting record sets (referenced **via their `@id`**), extracting data, performing basic exploratory analysis, and visualizing findings.

### Dataset Source
The dataset is defined by a Croissant schema at: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

In [ ]:
# Install mlcroissant in the current Jupyter environment
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. This step will fetch the Croissant schema and connect you to the dataset records and fields.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'
# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print basic metadata
print("Dataset Loaded!")
print(f"Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview
List and briefly inspect the available *record sets*, their fields, and their respective `@id` values as defined in the Croissant schema.

**Note:** All references to record sets and fields use their unique `@id` values for clarity and correctness.

In [ ]:
# List all record sets by @id and name
print("Record Sets available in this dataset:")
for record_set in dataset.metadata.recordSets:
    print(f"  - @id: {record_set['@id']}")
    print(f"    name: {record_set.get('name', '<no name>')}")
    print(f"    description: {record_set.get('description', '<no description>')}")
    print("    Fields:")
    for field in record_set['field']:
        fid = field['@id'] if isinstance(field, dict) else field
        print(f"      - {fid}")
    print("-")

## 3. Data Extraction
Load one or more record sets (tables) by their `@id`, and inspect their columns.

**We will use one of the main record sets of this dataset:**

For this example (from the FAIR² schema), let's use the main record set for protein abundance results. We'll reference it by its `@id`. *Check the previous cell's output for available `@id` values in your schema.*

In [ ]:
# You may need to customize this list if your schema includes different record sets
# We'll collect the main record set(s) as discovered above

# Gather record set @ids (edit if you wish to collect more)
record_set_ids = [rset['@id'] for rset in dataset.metadata.recordSets]
print(f"Record sets found: {record_set_ids}")

# Load all tables into pandas DataFrames indexed by their @id
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records from {record_set_id}")
    df = pd.DataFrame(list(dataset.records(record_set=record_set_id)))
    dataframes[record_set_id] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(df.head(2))

# For further analysis, pick the main record set (here, we simply use the first one discovered)
main_record_set_id = record_set_ids[0]
df = dataframes[main_record_set_id]
print(f"\nMain record set in use: {main_record_set_id}")
print(df.head())

## 4. Exploratory Data Analysis (EDA)
Let's analyze a numeric field from the main record set, such as protein abundance, peptide count, coverage, or molecular weight (MW). We'll reference the field **by its `@id`** as documented in Section 2.

Steps include filtering for high values, normalizing, and grouping if a suitable categorical variable exists (e.g., experimental condition, sample group).

> *Update field IDs below for your chosen field and group according to the schema overview above.*

In [ ]:
# Choose your field and group field by their `@id`
# Suggestions based on the description: 'coverage_percent', 'MW', 'peptide_count', 'abundance_sample_1', etc.
numeric_field_id = 'coverage_percent'  # e.g., '@id' of the coverage percent field in the main record set
group_field_id = 'sample_group'        # e.g., '@id' of a categorical field

# Print available columns if unsure
print("Available columns in main record set:", df.columns.tolist())

# Use fallback field name if the suggested IDs are not present
if numeric_field_id not in df.columns:
    # Try a more common field if available
    fallback_numeric_fields = [col for col in df.columns if col.lower() in ['mw', 'peptide_count', 'abundance', 'intensity']]
    if fallback_numeric_fields:
        numeric_field_id = fallback_numeric_fields[0]
        print(f"Switching to available numeric field: {numeric_field_id}")

# EDA step: filter high values (user can adjust threshold)
threshold = df[numeric_field_id].mean() if numeric_field_id in df.columns else 10

filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize the field (z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by group_field_id if exists
if group_field_id in df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Mean {numeric_field_id} by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize distributions or relationships between numeric fields. For example, plot the distribution of protein abundance or molecular weight, or a scatter plot of two variables.

We'll use `matplotlib` and `seaborn` for plotting.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the main numeric field
plt.figure(figsize=(8,5))
sns.histplot(df[numeric_field_id].dropna(), bins=30)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# If there's a second numeric field, show scatterplot
other_numeric_fields = [col for col in df.columns if col != numeric_field_id and pd.api.types.is_numeric_dtype(df[col])]
if other_numeric_fields:
    plt.figure(figsize=(8,6))
    sns.scatterplot(x=df[numeric_field_id], y=df[other_numeric_fields[0]])
    plt.xlabel(numeric_field_id)
    plt.ylabel(other_numeric_fields[0])
    plt.title(f"Scatter plot of {numeric_field_id} vs. {other_numeric_fields[0]}")
    plt.show()

## 6. Conclusion
- We've loaded the FAIR² protein dataset via `mlcroissant`, listed available record sets and fields **by their `@id`** for reproducibility, performed basic EDA and normalization, grouped and visualized data.
- For further analysis, consult the Croissant metadata for domain-specific field meanings, and use additional filtering, joining, or visualization as needed.

*Notebook template based on mlcroissant example and adapted for the FAIR² dataset with proper entity referencing via `@id` fields.*